<a href="https://colab.research.google.com/github/Moquiuti/LangGraph_Orquestrando_agentes_e_multiagentes/blob/main/Busca_ag%C3%AAntica_e_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q -U ddgs
!pip install -q -U tavily-python
!pip install -q -U langchain
!pip install -q -U langchain-google-genai
!pip install -q -U langgraph
!pip install -q -U selenium
!pip install -q -U beautifulsoup4
!pip install -q -U lxml
!pip install -q -U pandas
!pip install -q -U python-dotenv

In [3]:
!apt-get update -qq
!apt-get install -y -qq chromium-chromedriver

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package apparmor.
(Reading database ... 118252 files and directories currently installed.)
Preparing to unpack .../apparmor_3.0.4-2ubuntu2.5_amd64.deb ...
Unpacking apparmor (3.0.4-2ubuntu2.5) ...
Selecting previously unselected package squashfs-tools.
Preparing to unpack .../squashfs-tools_1%3a4.5-3build1_amd64.deb ...
Unpacking squashfs-tools (1:4.5-3build1) ...
Selecting previously unselected package udev.
Preparing to unpack .../udev_249.11-0ubuntu3.20_amd64.deb ...
Unpacking udev (249.11-0ubuntu3.20) ...
Selecting previously unselected package libfuse3-3:amd64.
Preparing to unpack .../libfuse3-3_3.10.5-1build1_amd64.deb ...
Unpacking libfuse3-3:amd64 (3.10.5-1build1) ...
Selecting previously unselected package snapd.
Preparing to unpack

In [4]:
import os
import re
import time
import pandas as pd

from dotenv import load_dotenv

from ddgs import DDGS
from tavily import TavilyClient

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By

In [5]:
try:
    from google.colab import userdata

    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

    print("Secrets carregados pelo Colab.")

except Exception:
    load_dotenv()
    print("Tentando carregar variáveis via arquivo .env.")

Secrets carregados pelo Colab.


In [6]:
#Aqui eu busquei alguns links de restaurantes no DuckDuckGo.

def busca_regular_duckduckgo(query: str, max_results: int = 10):
    resultados = []

    with DDGS() as ddgs:
        for item in ddgs.text(query, max_results=max_results):
            resultados.append({
                "titulo": item.get("title"),
                "link": item.get("href"),
                "descricao": item.get("body")
            })

    return resultados

In [7]:
query = "melhores restaurantes Campo Grande MS TripAdvisor"

resultados_ddg = busca_regular_duckduckgo(query, max_results=10)

df_ddg = pd.DataFrame(resultados_ddg)
df_ddg

,titulo,link,descricao
0,OS 10 MELHORES restaurantes: Campo Grande Atua...,https://www.tripadvisor.com.br/Restaurants-g30...,Melhores restaurantes em Campo Grande.Os vence...
1,Yallah aconchegante e com a melhor gastronomia...,https://www.tripadvisor.com/LocationPhotoDirec...,#3 of 657 restaurants in Campo Grande.Tripadvi...
2,Café y té Restaurantes de Campo Grande: Leé......,https://www.tripadvisor.com.ve/Restaurants-g30...,"Restaurantes Café y té en Campo Grande, Estado..."
3,Restaurante Campo Grande | TikTok,https://www.tiktok.com/discover/restaurante-ca...,Restaurantes Favoritos em Campo Grande - MS. D...
4,"Bom Almoço Restaurante, Campo Grande - City Ce...",https://www.tripadvisor.com.my/Restaurant_Revi...,Campo Grande Restaurants.Tripadvisor gives a T...
5,Campo Grande Turismo - Información turística s...,https://www.tripadvisor.com.ar/Tourism-g303369...,"Campo Grande, MS.Tripadvisor LLC no ofrece nin..."
6,Os melhores restaurantes em Campo Grande - Con...,https://blog.123milhas.com/os-melhores-restaur...,Com influências e toques gastronômicos de dife...
7,Fantastic place. Brilliant atmosphere - Review...,https://www.tripadvisor.com.ph/ShowUserReviews...,Pizzaria Giuseppe: Fantastic place. Brilliant ...
8,Top 10 melhores restaurantes: campo grande - ms,https://www.youtube.com/watch?v=6L0rZdjWU_Y,О сервисе Прессе Авторские права Связаться с н...
9,Link to instagram.com,https://www.instagram.com/tomate_restaurantes/,The site owner hides the web page description.


In [10]:
def busca_agentica_tavily(query: str, max_results: int = 5):
    resposta = client.search(
        query=query,
        search_depth="advanced",
        max_results=max_results,
        include_answer=True,
        include_raw_content=False
    )

    resultados = resposta.get("results", [])

    itens = []

    for item in resultados:
        itens.append({
            "titulo": item.get("title"),
            "url": item.get("url"),
            "conteudo": item.get("content"),
            "score": item.get("score")
        })

    return {
        "resposta": resposta.get("answer"),
        "resultados": itens
    }

In [32]:
query_tavily = """
Encontre a URL mais relevante do TripAdvisor com lista de restaurantes em Campo Grande MS Brasil.
Priorize páginas do domínio tripadvisor.com.br ou tripadvisor.com.
"""

resultado_tavily = busca_agentica_tavily(query_tavily, max_results=8)

print("Resposta agêntica:")
print(resultado_tavily["resposta"])

df_tavily = pd.DataFrame(resultado_tavily["resultados"])
df_tavily

Resposta agêntica:
The most relevant TripAdvisor URL for Campo Grande MS restaurants is https://www.tripadvisor.com.br/Restaurants-g303610-Campo_Grande_Mato_Grosso_do_Sul.state.html. This page lists top dining options in the city.


,titulo,url,conteudo,score
0,The 50 best restaurants to have dinner in Camp...,https://wanderlog.com/list/geoCategory/30970/b...,## Create your ultimate travel itinerary\n\nPl...,0.580376
1,Melhores restaurantes de Campo Grande MS: Não ...,https://todepassagem.clickbus.com.br/sabores-d...,Casa do Peixe em Campo Grande\n\nFoto Merament...,0.542743
2,Lista de Empresas de Restaurantes com mais de ...,https://www.econodata.com.br/empresas/ms-campo...,Adicionar\n\n| | CNPJ e Nome | Endereço | CNA...,0.501685
3,Lista de Empresas de Restaurantes com mais de ...,https://www.econodata.com.br/empresas/ms-campo...,Adicionar\n\n| | CNPJ e Nome | Endereço | CNA...,0.498023
4,Lista das Melhores Salgadinhos em Campo Grande...,https://empresafone.com.br/lista-empresas/salg...,| Marketing | Somos um diretório de negócios g...,0.293998
5,Lista das Melhores Festas - Artigos em Campo G...,https://empresafone.com.br/lista-empresas/fest...,Este website utiliza cookies para garantir a s...,0.232572
6,Lista das Melhores Apart-Hotéis em Campo Grand...,https://empresafone.com.br/lista-empresas/apar...,| Categoria | Descriçāo | Status |\n --- \n| N...,0.139831
7,Lista das Melhores Portas De Enrolar em Campo ...,https://empresafone.com.br/lista-empresas/port...,| Categoria | Descriçāo | Status |\n --- \n| N...,0.131787


In [12]:
def selecionar_url_tripadvisor(resultados):
    for item in resultados:
        url = item.get("url", "")

        if "tripadvisor" in url.lower():
            return url

    return None

In [33]:
url_tripadvisor = selecionar_url_tripadvisor(resultado_tavily["resultados"])

print("URL selecionada:")
print(url_tripadvisor)

URL selecionada:
None


In [37]:
def criar_driver():
    chrome_options = Options()

    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--lang=pt-BR")

    chrome_options.binary_location = "/usr/bin/chromium-browser"

    # Corrected path for chromedriver
    service = Service("/usr/lib/chromium-browser/chromedriver")

    driver = webdriver.Chrome(
        service=service,
        options=chrome_options
    )

    return driver

In [15]:
def obter_html_com_selenium(url: str, tempo_espera: int = 5):
    driver = criar_driver()

    try:
        driver.get(url)
        time.sleep(tempo_espera)

        html = driver.page_source

        return html

    finally:
        driver.quit()

In [35]:
if url_tripadvisor:
    html = obter_html_com_selenium(url_tripadvisor, tempo_espera=7)
    print(html[:1000])
else:
    print("Nenhuma URL do TripAdvisor foi encontrada.")

Nenhuma URL do TripAdvisor foi encontrada.


In [17]:
def limpar_texto(texto):
    if not texto:
        return None

    return re.sub(r"\s+", " ", texto).strip()

In [18]:
def extrair_restaurantes_tripadvisor(html: str, limite: int = 10):
    soup = BeautifulSoup(html, "lxml")

    restaurantes = []

    seletores_candidatos = [
        "[data-test-target='restaurants-list'] a",
        "a[href*='Restaurant_Review']",
        "div[data-automation='cardTitle']",
        "div[class*='restaurant']",
    ]

    links_restaurantes = soup.select("a[href*='Restaurant_Review']")

    urls_vistas = set()

    for link in links_restaurantes:
        nome = limpar_texto(link.get_text(" "))

        href = link.get("href")

        if not nome or not href:
            continue

        if href in urls_vistas:
            continue

        urls_vistas.add(href)

        if href.startswith("/"):
            href = "https://www.tripadvisor.com.br" + href

        container = link.find_parent(["div", "section", "article"])

        texto_container = limpar_texto(container.get_text(" ")) if container else ""

        avaliacao = None
        tipo = None
        faixa_preco = None

        if texto_container:
            match_avaliacao = re.search(r"(\d,\d|\d\.\d|\d)\s*(?:de\s*5|/5)?", texto_container)
            if match_avaliacao:
                avaliacao = match_avaliacao.group(1)

            tipos_possiveis = [
                "Brasileira", "Italiana", "Japonesa", "Churrasco", "Pizza",
                "Sul-americana", "Contemporânea", "Café", "Bar", "Pub",
                "Fast food", "Saudável", "Frutos do mar", "Steakhouse"
            ]

            tipos_encontrados = [
                tipo_item for tipo_item in tipos_possiveis
                if tipo_item.lower() in texto_container.lower()
            ]

            if tipos_encontrados:
                tipo = ", ".join(tipos_encontrados)

            match_preco = re.search(r"(R\$\s?\d+[^A-Za-z]{0,20}R\$\s?\d+|R\$\s?[\d\.]+)", texto_container)
            if match_preco:
                faixa_preco = limpar_texto(match_preco.group(1))

        restaurantes.append({
            "nome": nome,
            "avaliacao": avaliacao,
            "tipo": tipo,
            "faixa_preco": faixa_preco,
            "url": href
        })

        if len(restaurantes) >= limite:
            break

    return restaurantes

In [20]:
#scraping em uma página mais simples

def extrair_restaurantes_por_busca_regular(cidade: str, max_results: int = 10):
    query = f"restaurantes {cidade} avaliação tipo faixa de preço"

    resultados = busca_regular_duckduckgo(query, max_results=max_results)

    restaurantes = []

    for item in resultados:
        titulo = item.get("titulo")
        link = item.get("link")
        descricao = item.get("descricao") or ""

        restaurantes.append({
            "nome": titulo,
            "avaliacao": None,
            "tipo": None,
            "faixa_preco": None,
            "descricao": descricao,
            "url": link
        })

    return restaurantes

In [21]:
fallback_restaurantes = extrair_restaurantes_por_busca_regular(
    "Campo Grande MS",
    max_results=10
)

pd.DataFrame(fallback_restaurantes)

,nome,avaliacao,tipo,faixa_preco,descricao,url
0,"LÁ IN CASA RESTAURANTE, Campo Grande - Comentá...",None,None,None,Tipos de refeições. Almoço. PREÇO.Faça uma ava...,https://www.tripadvisor.com.br/Restaurant_Revi...
1,Três Restaurantes Imperdíveis em Campo Grande ...,None,None,None,TOP 3 ORIENTAIS DE CG/MS Esse é o nosso rankin...,https://www.tiktok.com/@casalporcampogrande/vi...
2,"Restaurante Fogão de Minas - Centro, Campo Gra...",None,None,None,"Aquino Centro, Campo Grande - MS. 55 (67) 3325...",https://www.maisdireto.com.br/comida-mineira/c...
3,Anésia Restaurante Coronel Antonino Campo Gran...,None,None,None,"Antonino, Campo Grande - MS, 79010-000, Brazil...",https://www.benditoguia.com.br/empresa/anesia-...
4,TENDA BAR E RESTAURANTE em Campo Grande MS,None,None,None,Campo Grande - MS. Descri��o. Caracter�sticas....,https://www.ferias.tur.br/empresa/7831/tendaba...
5,fogo em restaurante campo grande ms,None,None,None,,https://www.youtube.com/watch?v=4PG3mxHMEmU
6,Roast Mania - Assados na Brasa e Marmitex Deli...,None,None,None,"Campo Grande, Mato Grosso do Sul /. Roast Mani...",https://restaurantguru.com.br/Frango-na-Brasa-...
7,Link to facebook.com,None,None,None,The site owner hides the web page description.,https://www.facebook.com/fogaodelenhams/
8,Link to instagram.com,None,None,None,The site owner hides the web page description.,https://www.instagram.com/tomate_restaurantes/
9,Google Карты,None,None,None,"Найти информацию о местных компаниях, посмотре...",https://maps.google.com/


In [22]:
#tudo em uma execução

def executar_pipeline_restaurantes(cidade: str):
    print("1. Executando busca regular com DuckDuckGo...")

    query_regular = f"melhores restaurantes {cidade} TripAdvisor"
    resultados_regulares = busca_regular_duckduckgo(query_regular, max_results=10)

    print(f"Busca regular retornou {len(resultados_regulares)} resultados.")

    print("\n2. Executando busca agêntica com Tavily...")

    query_agentica = f"""
    Encontre a URL mais relevante do TripAdvisor com lista de restaurantes em {cidade}.
    Priorize páginas com lista de restaurantes, avaliações e informações de preço.
    """

    resultado_agentico = busca_agentica_tavily(query_agentica, max_results=8)

    url = selecionar_url_tripadvisor(resultado_agentico["resultados"])

    print("URL selecionada pela busca agêntica:")
    print(url)

    if not url:
        print("\nNenhuma URL do TripAdvisor encontrada. Usando fallback com DuckDuckGo.")
        dados = extrair_restaurantes_por_busca_regular(cidade)
        return pd.DataFrame(dados)

    print("\n3. Carregando HTML com Selenium...")

    html = obter_html_com_selenium(url, tempo_espera=7)

    print("\n4. Extraindo dados com BeautifulSoup...")

    dados = extrair_restaurantes_tripadvisor(html, limite=10)

    if not dados:
        print("\nNão foi possível extrair dados do HTML. Usando fallback.")
        dados = extrair_restaurantes_por_busca_regular(cidade)

    return pd.DataFrame(dados)

In [38]:
#ultimo teste

df_final = executar_pipeline_restaurantes("Campo Grande MS Brasil")
df_final

1. Executando busca regular com DuckDuckGo...
Busca regular retornou 10 resultados.

2. Executando busca agêntica com Tavily...
URL selecionada pela busca agêntica:
https://www.tripadvisor.com.br/Restaurants-g303369-Campo_Grande_State_of_Mato_Grosso_do_Sul.html

3. Carregando HTML com Selenium...


WebDriverException: Message: Service /usr/lib/chromium-browser/chromedriver unexpectedly exited. Status code was: 1


In [24]:
from google.colab import files

files.download("restaurantes_extraidos.csv")

FileNotFoundError: Cannot find file: restaurantes_extraidos.csv

In [25]:
import os

print(os.listdir())

['.config', 'sample_data']


In [26]:
try:
    print(df_final.head())
except NameError:
    print("df_final ainda não foi criado.")

df_final ainda não foi criado.


In [28]:
import os
from tavily import TavilyClient

try:
    from google.colab import userdata
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
    print("TAVILY_API_KEY carregada pelo Colab Secrets.")
except Exception:
    print("Não foi possível carregar pelo Colab Secrets. Verifique se está rodando no Colab.")

tavily_api_key = os.getenv("TAVILY_API_KEY")

if not tavily_api_key:
    raise ValueError("TAVILY_API_KEY não encontrada. Configure a chave nos Secrets do Colab.")

client = TavilyClient(api_key=tavily_api_key)

print("Cliente Tavily inicializado com sucesso.")

TAVILY_API_KEY carregada pelo Colab Secrets.
Cliente Tavily inicializado com sucesso.


In [41]:
df_final = executar_pipeline_restaurantes("Campo Grande MS Brasil")
df_final

1. Executando busca regular com DuckDuckGo...
Busca regular retornou 10 resultados.

2. Executando busca agêntica com Tavily...
URL selecionada pela busca agêntica:
https://www.tripadvisor.com.br/Restaurants-g303369-Campo_Grande_State_of_Mato_Grosso_do_Sul.html

3. Carregando HTML com Selenium...


WebDriverException: Message: Service /usr/lib/chromium-browser/chromedriver unexpectedly exited. Status code was: 1


In [42]:
!which chromium
!which chromium-browser
!which google-chrome
!which chromedriver
!ls -l /usr/bin/chromedriver
!ls -l /usr/lib/chromium-browser/chromedriver

/usr/bin/chromium-browser
/usr/bin/chromedriver
-rwxr-xr-x 1 root root 312 Sep 18  2020 /usr/bin/chromedriver
lrwxrwxrwx 1 root root 22 Oct 17  2022 /usr/lib/chromium-browser/chromedriver -> ../../bin/chromedriver


In [43]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import shutil

def criar_driver():
    chrome_options = Options()

    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--disable-software-rasterizer")
    chrome_options.add_argument("--remote-debugging-port=9222")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--lang=pt-BR")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-background-networking")
    chrome_options.add_argument("--disable-default-apps")
    chrome_options.add_argument("--disable-sync")

    chromium_path = (
        shutil.which("chromium")
        or shutil.which("chromium-browser")
        or shutil.which("google-chrome")
    )

    chromedriver_path = shutil.which("chromedriver")

    print("Chromium encontrado em:", chromium_path)
    print("ChromeDriver encontrado em:", chromedriver_path)

    if not chromium_path:
        raise Exception("Chromium/Chrome não encontrado no ambiente.")

    if not chromedriver_path:
        raise Exception("ChromeDriver não encontrado no ambiente.")

    chrome_options.binary_location = chromium_path

    service = Service(chromedriver_path)

    driver = webdriver.Chrome(
        service=service,
        options=chrome_options
    )

    return driver

In [48]:
driver = criar_driver()
driver.get("https://www.google.com")
print(driver.title)
driver.quit()

Chromium encontrado em: /usr/bin/chromium-browser
ChromeDriver encontrado em: /usr/bin/chromedriver


WebDriverException: Message: Service /usr/bin/chromedriver unexpectedly exited. Status code was: 1


In [47]:
!apt-get update -qq
!apt-get install -y -qq chromium chromium-driver

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
E: Package 'chromium' has no installation candidate


In [49]:
df_final.to_csv("restaurantes_extraidos.csv", index=False, encoding="utf-8-sig")
print("Arquivo restaurantes_extraidos.csv criado com sucesso.")

NameError: name 'df_final' is not defined

Substituindo a parte do Selenium por uma versão com request, que tenta baixar o html direto. Se o site bloquear ele cai no fallback...
vou fazer isso porque decidi não perder mais tempo tentando forçar Selenium no Colab.
Devido a isso eu irei concluir a atividade com fallback usando DuckDuckGo + Tavily + BeautifulSoup/requests, pelo fato do Selenium ter falhado por limitação do ambiente.

Devido o fato da parte importante deste modulo ter sido alcançado:
busca regular
busca agêntica
seleção de URL
tentativa de scraping
tratamento de falha
fallback funcional

In [51]:
!pip install -q -U requests beautifulsoup4 lxml pandas

In [52]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re

In [53]:
def obter_html_com_requests(url: str):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
    }

    resposta = requests.get(url, headers=headers, timeout=20)

    if resposta.status_code != 200:
        raise Exception(f"Erro ao acessar página. Status code: {resposta.status_code}")

    return resposta.text

In [54]:
def executar_pipeline_restaurantes(cidade: str):
    print("1. Executando busca regular com DuckDuckGo...")

    query_regular = f"melhores restaurantes {cidade} TripAdvisor"
    resultados_regulares = busca_regular_duckduckgo(query_regular, max_results=10)

    print(f"Busca regular retornou {len(resultados_regulares)} resultados.")

    print("\n2. Executando busca agêntica com Tavily...")

    query_agentica = f"""
    Encontre a URL mais relevante do TripAdvisor com lista de restaurantes em {cidade}.
    Priorize páginas com lista de restaurantes, avaliações e informações de preço.
    """

    resultado_agentico = busca_agentica_tavily(query_agentica, max_results=8)

    url = selecionar_url_tripadvisor(resultado_agentico["resultados"])

    print("URL selecionada pela busca agêntica:")
    print(url)

    if not url:
        print("\nNenhuma URL do TripAdvisor encontrada. Usando fallback com DuckDuckGo.")
        dados = extrair_restaurantes_por_busca_regular(cidade)
        return pd.DataFrame(dados)

    try:
        print("\n3. Carregando HTML com requests...")

        html = obter_html_com_requests(url)

        print("\n4. Extraindo dados com BeautifulSoup...")

        dados = extrair_restaurantes_tripadvisor(html, limite=10)

        if not dados:
            print("\nNão foi possível extrair dados do HTML. Usando fallback.")
            dados = extrair_restaurantes_por_busca_regular(cidade)

    except Exception as erro:
        print("\nErro ao acessar ou processar HTML.")
        print("Detalhe do erro:", erro)
        print("\nUsando fallback com DuckDuckGo.")

        dados = extrair_restaurantes_por_busca_regular(cidade)

    return pd.DataFrame(dados)

In [56]:
df_final = executar_pipeline_restaurantes("Campo Grande MS Brasil")
df_final

1. Executando busca regular com DuckDuckGo...
Busca regular retornou 10 resultados.

2. Executando busca agêntica com Tavily...
URL selecionada pela busca agêntica:
https://www.tripadvisor.com.br/Restaurants-g303369-Campo_Grande_State_of_Mato_Grosso_do_Sul.html

3. Carregando HTML com requests...

Erro ao acessar ou processar HTML.
Detalhe do erro: Erro ao acessar página. Status code: 403

Usando fallback com DuckDuckGo.


,nome,avaliacao,tipo,faixa_preco,descricao,url
0,OS MELHORES restaurantes: Campo Grande Atualiz...,None,None,None,Melhores restaurantes em Campo Grande. Restaur...,https://www.tripadvisor.com.br/Restaurants-g16...
1,Três Restaurantes Imperdíveis em Campo Grande ...,None,None,None,Gostamos demais Faixa de preço: ~R$100-150 por...,https://www.tiktok.com/@casalporcampogrande/vi...
2,Onde comer em Campo Grande: seleção dos melhor...,None,None,None,Onde comer bem em Campo Grande: aqui você enco...,https://blogmaladeviagem.com.br/onde-comer-cam...
3,Fim da escala 6x1: consumidor rejeitará mudanç...,None,None,None,Consumidor rejeitará fim da escala 6x1 quando ...,https://www.bbc.com/portuguese/articles/c4geze...
4,Melhores hotéis econômicos em Campo Grande a p...,None,None,None,Avaliação. Preço. Serviços incluídos.Campo Gra...,https://www.kayak.com.br/Campo-Grande-Hoteis_E...
5,"Campo Grande - MS, o que fazer por aqui - com....",None,None,None,Estou indo para o Mato Grosso do Sul e gostari...,https://www.mochileiros.com/topic/17024-campo-...
6,instacarro.com,None,None,None,"Se o vendedor queria achar um preço justo, tin...",https://www.instacarro.com/
7,Hortas ocupam os espaços urbanos e se tornam o...,None,None,None,"Campo Grande/MS, sábado, 7 de março de 2026.– ...",https://www.campograndenoticias.com.br/hortas-...
8,Rihanna - Diamonds - YouTube,None,None,None,Джем – Rihanna - Don't Stop The Music Live in ...,https://www.youtube.com/watch?v=lWA2pjMjpBs
9,Link to facebook.com,None,None,None,The site owner hides the web page description.,https://www.facebook.com/marmitteiroslight/


In [57]:
df_final.to_csv("restaurantes_extraidos.csv", index=False, encoding="utf-8-sig")
print("Arquivo restaurantes_extraidos.csv criado com sucesso.")

Arquivo restaurantes_extraidos.csv criado com sucesso.


In [58]:
from google.colab import files
files.download("restaurantes_extraidos.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>